# E6 privacy and controlled memorization runs

Run after the full CAB/BBQ notebook finishes, on a fresh NVIDIA L4 Colab runtime. This notebook keeps the original proposal's six PII conditions, three-epoch Phi QLoRA canary adapter, and base/adapter membership attacks. It writes private row-level results and aggregate summaries to the mounted Drive. Run cells in order. If Colab disconnects, repeat setup and the interrupted cell using the **same Git commit**; training resumes at the last verified epoch and attack scoring resumes at a verified 100-record part. Do not put secrets, raw PII, canaries, or private outputs in Git.


## 1. Mount Drive and install the pinned environment


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, subprocess, sys

drive.mount("/content/drive")
SHARED = Path("/content/drive/MyDrive/CMPE253-Project").resolve()
assert SHARED.is_dir(), f"Missing shared project folder: {SHARED}"
REPO = Path("/content/cmpe253-llm-demographic-bias-audit")
URL = "https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git"
if (REPO / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
else:
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip(), "Clean Git checkout required"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["make", "setup-colab"], cwd=REPO, check=True)
GPU = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
assert GPU == "NVIDIA L4", f"Expected NVIDIA L4; found {GPU!r}"
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Ready:", GPU, COMMIT, SHARED)


## 2. Renew the versioned data gate and check redaction

The updated `LLM_Bias_Audit_PRD.md` must already be copied to the Drive project root. The v4 receipt leaves the full-run v3 receipt intact.


In [ ]:
subprocess.run(["make", "ai-review-gate", f"ROOT={SHARED}"], cwd=REPO, check=True)
GATE = SHARED / "manifests/data_gate_approval_v4.json"
assert GATE.is_file()
subprocess.run(["make", "privacy-controls-smoke"], cwd=REPO, check=True)
print("Data gate and pinned redaction smoke passed:", GATE)


## 3. M1: raw, input-redacted, and output-filtered PII conditions

Each cell contains exactly 200 held-out fictional records. A completed condition has a 200-row inference manifest, zero errors, and an L4 environment. A P2 filter receipt is also required.


In [ ]:
for run_id in ("PII-M1-P0", "PII-M1-P1", "PII-M1-P2"):
    print("Starting", run_id, flush=True)
    subprocess.run(["make", "run-pii", f"ROOT={SHARED}", f"RUN_ID={run_id}", "SPLIT=test"], cwd=REPO, check=True)
    manifest = json.loads((SHARED / "manifests/inference" / f"{run_id}.json").read_text())
    assert manifest["status"] == "complete"
    assert manifest["expected_rows"] == manifest["completed_rows"] == manifest["successful_rows"] == 200
    assert manifest["error_summary"] == {} and manifest["environment"]["gpu"]["name"] == "NVIDIA L4"
    assert manifest["code_commit"] == COMMIT
    if run_id.endswith("P2"):
        assert (SHARED / "manifests/privacy" / f"{run_id}_filter.json").is_file()
    print("Completed", run_id, "rows", manifest["completed_rows"], flush=True)


## 4. M2: the same three PII conditions


In [ ]:
for run_id in ("PII-M2-P0", "PII-M2-P1", "PII-M2-P2"):
    print("Starting", run_id, flush=True)
    subprocess.run(["make", "run-pii", f"ROOT={SHARED}", f"RUN_ID={run_id}", "SPLIT=test"], cwd=REPO, check=True)
    manifest = json.loads((SHARED / "manifests/inference" / f"{run_id}.json").read_text())
    assert manifest["status"] == "complete"
    assert manifest["expected_rows"] == manifest["completed_rows"] == manifest["successful_rows"] == 200
    assert manifest["error_summary"] == {} and manifest["environment"]["gpu"]["name"] == "NVIDIA L4"
    assert manifest["code_commit"] == COMMIT
    if run_id.endswith("P2"):
        assert (SHARED / "manifests/privacy" / f"{run_id}_filter.json").is_file()
    print("Completed", run_id, "rows", manifest["completed_rows"], flush=True)


## 5. Score privacy exposure and the frozen H5 utility test

This produces measured aggregate results. The human helpfulness audit is a separate pending validity check; this cell makes no human-rating claim.


In [ ]:
subprocess.run(["make", "score-pii", f"ROOT={SHARED}"], cwd=REPO, check=True)
subprocess.run(["make", "score-h5", f"ROOT={SHARED}"], cwd=REPO, check=True)
EXPOSURE = SHARED / "aggregate_results/privacy_exposure_summary.json"
H5 = SHARED / "aggregate_results/privacy_h5.json"
assert EXPOSURE.is_file() and H5.is_file()
print("Exposure summary:", EXPOSURE)
print("H5 summary:", H5)


## 6. Train the controlled Phi QLoRA adapter

This is the proposal's three-epoch memorization experiment. It checkpoints after each 3,200-presentation epoch. If interrupted mid-epoch, rerun this cell; only that unfinished epoch is repeated.


In [ ]:
subprocess.run(["make", "train-canary", f"ROOT={SHARED}"], cwd=REPO, check=True)
EPOCHS = [SHARED / "manifests/canary_training" / f"epoch_{number}.json" for number in (1, 2, 3)]
receipts = [json.loads(path.read_text()) for path in EPOCHS]
assert [row["epoch"] for row in receipts] == [1, 2, 3]
assert all(row["presentations"] == 3200 and row["context"]["code_commit"] == COMMIT for row in receipts)
print("Three verified training epochs:", EPOCHS[-1])


## 7. Run the base and trained-adapter canary attacks

Both use the same pinned 4-bit Phi base. Each mode scores 300 member canaries, 500 threshold-calibration non-members, and 500 held-out evaluation non-members. The attack runner saves 100-record parts.


In [ ]:
for mode in ("base", "adapter"):
    print("Starting canary attack", mode, flush=True)
    subprocess.run(["make", "attack-canary", f"ROOT={SHARED}", f"MODE={mode}"], cwd=REPO, check=True)
    manifest_path = SHARED / "manifests/canary_attack" / f"{mode}.json"
    output_path = SHARED / "private_outputs/canary_attack" / f"{mode}.jsonl"
    aggregate_path = SHARED / "aggregate_results" / f"canary_attack_{mode}.json"
    manifest = json.loads(manifest_path.read_text())
    assert manifest["mode"] == mode and manifest["code_commit"] == COMMIT
    assert hashlib.sha256(output_path.read_bytes()).hexdigest() == manifest["private_output_sha256"]
    assert len(output_path.read_text().splitlines()) == 1300 and aggregate_path.is_file()
    print("Completed canary attack", mode, "records", 1300, flush=True)


## 8. Check ordinary-task utility before and after the adapter

This supports interpretation of the memorization result; the adapter is not a fairness mitigation.


In [ ]:
for mode in ("base", "adapter"):
    subprocess.run(["make", "score-adapter-utility", f"ROOT={SHARED}", f"MODE={mode}"], cwd=REPO, check=True)
subprocess.run(["make", "compare-adapter-utility", f"ROOT={SHARED}"], cwd=REPO, check=True)
UTILITY = SHARED / "aggregate_results/adapter_utility_comparison.json"
assert UTILITY.is_file()
print("Adapter utility summary:", UTILITY)


## 9. Verify artifact presence and report completion

Send the printed paths and any first failure to Codex. Do not publish private row-level outputs. The blinded 300-rating helpfulness audit is not performed by this notebook and must not be described as complete.


In [ ]:
required = [GATE, EXPOSURE, H5, EPOCHS[-1], UTILITY]
required += [SHARED / "aggregate_results" / f"canary_attack_{mode}.json" for mode in ("base", "adapter")]
assert all(path.is_file() for path in required)
for path in required:
    print(path.relative_to(SHARED), hashlib.sha256(path.read_bytes()).hexdigest())
print("Privacy exposure, H5, canary training, attacks, and adapter utility artifacts are present.")
